# Validation score separation

Why does one detector produce very different false-positive rates across recordings?
This analysis compares the frozen model's scores for normal and anomalous observations
in **validation groups only**. It does not fit a model, select a new threshold, or score
the test recordings. The existing modeling notebook remains the exploratory workspace.

Install the project and download the training/validation data using the README.
A trusted local artifact from `python -m forge train` is required. Paths below resolve
from the package location, so the repository can live on any drive or in any folder.

In [ ]:
from IPython.display import display

from forge.ml.diagnostics import (
    analyze_validation,
    distribution_figure,
    export_diagnostics,
    histograms,
)

groups, scores, summary, provenance = analyze_validation()
print(f"Model run: {provenance['run_id']}")
print(f"Frozen threshold: {provenance['threshold']:.6f}")
print(f"Partition: {provenance['partition']}")
print(f"Policy: {provenance['policy']}")

## 1. Alert rates and score ranking

**FPR** is the fraction of normal observations that trigger an alert after the
three-reading persistence rule. **Recall** is the fraction of anomalous observations
that trigger one. These are point rates, not incident counts.

**ROC AUC** measures raw score ordering independently of this threshold: 1 is perfect
separation, 0.5 is chance-level ordering, and below 0.5 means the score ordering is
reversed more often than correct. These are descriptive values, without uncertainty
intervals for the temporally dependent observations.

Average precision also depends on class prevalence, included for context. High recall
with high FPR can simply mean almost everything is being flagged.

In [ ]:
display(
    summary[
        [
            "normal_rows",
            "anomaly_rows",
            "fpr",
            "recall",
            "roc_auc",
            "average_precision",
            "anomaly_prevalence",
        ]
    ]
    .sort_values("fpr", ascending=False)
    .round(4)
)

## 2. Normal versus anomalous score distributions

Every panel uses the same score bins and axis scales. Each class is normalized to
100% within its group so a larger class does not dominate the chart. Teal marks
normal annotations; orange marks anomalous annotations. The dashed line is the frozen
threshold. Larger scores mean the detector considers a reading more unusual.

Look for two patterns: both distributions lying above the line indicates many
threshold exceedances; overlapping or reversed distributions indicate a ranking
problem that moving a threshold may not resolve. Histograms describe raw scores,
so their mass above the line is not the persistent-alert FPR shown in the titles.

In [ ]:
binned = histograms(groups, scores)
figure = distribution_figure(binned, summary, provenance)
display(figure)

## 3. Check the picture against numerical summaries

The 10th, 50th (median), and 90th percentiles describe the central score ranges.
`normal_above_threshold` is the raw fraction above the threshold; `fpr` includes
persistence. Persistence can suppress isolated exceedances but cannot fix sustained
high scores throughout normal operation.

In [ ]:
display(
    summary[
        [
            "normal_p10",
            "normal_p50",
            "normal_p90",
            "anomaly_p10",
            "anomaly_p50",
            "anomaly_p90",
            "normal_above_threshold",
            "fpr",
            "anomaly_above_threshold",
            "recall",
        ]
    ].round(4)
)

## 4. Interpretation of the recorded baseline

For run `20261007T003523Z-e78e7b17`, `other/1` has ROC AUC 0.175: its normal-labeled
observations generally rank above its anomalous observations. The shared threshold
also sits below both medians. This is more than a threshold-placement problem.

`other/10` and `valve2/1` have AUC around 0.786 and 0.782. Their anomaly scores
generally rank higher, but normal score medians exceed the shared threshold. A
different operating point may reduce false alerts at a recall cost; that tradeoff
has not been selected or applied here.

`other/7` has AUC 0.996, FPR 0.5%, and recall 99.1%. The failure is therefore not
uniform across the `other` family. Shifts in the score distributions of normal
readings are directly visible. Their physical causes remain unverified.

These findings describe the listed run. If the run printed above differs, interpret
its newly computed table before reusing these numbers. See
[`docs/validation-diagnostics.md`](../docs/validation-diagnostics.md) for the saved
findings and the boundary between diagnosis of model behavior and new model selection.

## 5. Export the figures

The next cell writes a standalone interactive HTML chart, a PNG, a CSV table, and
aggregate JSON under the ignored `reports/figures/validation-<run>/` directory.
It does not change the frozen model or public held-out results. Notebook outputs
are kept out of Git; rerun locally to display the figures.

In [ ]:
output_directory = export_diagnostics(groups, scores, summary, provenance)
print(output_directory)